# Занятие 40. Практика: бустинг в реальных библиотеках

**Тема:** предсказание прогрессирования диабета по медицинским признакам (реальный датасет `load_diabetes`).  
**Модельный фокус:** градиентный бустинг для регрессии.  
**Библиотеки:** `sklearn`, `xgboost`, `lightgbm`, `catboost` доступны в учебной среде и сравниваются обязательно.


## Легенда

Вы работаете в аналитической команде медицинского сервиса. Нужно предсказывать численный индекс прогрессирования заболевания через год, чтобы врачам было проще выделять пациентов с высоким риском.

Почему это хороший кейс для бустинга:
- задача регрессии на табличных данных;
- признаки имеют нелинейные связи;
- важны качество, скорость обучения и интерпретация;
- в реальных проектах обычно сравнивают несколько библиотек бустинга, а не одну.


## Что нужно сделать

1. Подготовить данные и baseline.
2. Разделить данные на train / validation / test.
3. Обучить и сравнить `GradientBoostingRegressor` и `HistGradientBoostingRegressor` из `sklearn`.
4. Обязательно сравнить внешние библиотеки (`xgboost`, `lightgbm`, `catboost`).
5. Выбрать модель по validation, а test использовать один раз — для финальной проверки выбранного решения.
6. Сформулировать практические выводы: когда какую библиотеку выбирать, какие у неё плюсы и минусы.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 0 | Загрузка и split | 3 |
| 1 | Baseline и метрики | 3 |
| 2 | Sklearn GradientBoostingRegressor | 4 |
| 3 | Подбор гиперпараметров для GBDT | 5 |
| 4 | HistGradientBoostingRegressor и early stopping | 4 |
| 5 | Внешние библиотеки бустинга | 5 |
| 6 | Единый рейтинг моделей | 2 |
| 7 | Преимущества и недостатки библиотек | 2 |
| 8 | Финальный инженерный вывод | 2 |
| | **Итого** | **30** |


---
## Среда и библиотеки

В учебной среде `xgboost`, `lightgbm` и `catboost` уже установлены. Если вы запускаете блокнот дома и импорт ниже падает, установите их отдельно перед занятием.


In [ ]:
# В учебной среде установка не нужна.
# Если запускаете дома и библиотек нет, выполните в терминале:
# pip install xgboost lightgbm catboost


In [ ]:
import importlib
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25  # 25% от train_val ≈ 20% от всех данных


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_regression(model_name, y_true, y_pred):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


---
## Задание 0. Загрузка и split — **3 балла**

**Сделайте:**
- загрузите датасет `load_diabetes(as_frame=True)`;
- соберите `X` и `y`;
- сначала отделите финальный `test`;
- из оставшейся части выделите `validation`;
- выведите размеры train/validation/test и первые 5 строк `X_train`.

### Подробные критерии (для проверки LLM)
- **1 балл:** датасет загружен через `load_diabetes(as_frame=True)`, переменные `X` и `y` созданы корректно.
- **1 балл:** получены `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test`; test не используется для выбора модели.
- **1 балл:** выведены размеры всех выборок и показаны первые 5 строк `X_train`.

### Снижение баллов
- Использован другой датасет или `load_diabetes` без `as_frame=True` → минус **1.0**.
- Нет отдельной validation или test → минус **1.0**.
- Не выведены размеры выборок или отсутствует `X_train.head()` → минус **0.5** за каждый пропуск.


In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data.copy()
y = diabetes.target.copy()

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=VAL_SIZE, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_val:", y_val.shape, "y_test:", y_test.shape)
X_train.head()


---
## Задание 1. Baseline и метрики — **3 балла**

**Сделайте:**
- обучите `DummyRegressor(strategy='mean')` на train;
- посчитайте MAE, RMSE, R2 на validation;
- сохраните результат в `baseline_metrics`.

### Подробные критерии (для проверки LLM)
- **1 балл:** baseline-модель обучена корректно.
- **1 балл:** метрики MAE/RMSE/R2 посчитаны на validation.
- **1 балл:** результат сохранён в `baseline_metrics` (словарь/таблица).

### Снижение баллов
- `DummyRegressor(strategy='mean')` не обучен или заменён другой baseline-моделью → минус **1.0**.
- Метрики посчитаны не на validation (`X_val`, `y_val`) → минус **1.0**.
- Результат не сохранён в `baseline_metrics` или отсутствует одна из метрик (MAE/RMSE/R2) → минус **0.5** за каждый пропуск.


In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_val)
baseline_metrics = evaluate_regression("DummyRegressor(mean)", y_val, y_pred_dummy)

baseline_metrics


---
## Задание 2. Sklearn GradientBoostingRegressor — **4 балла**

**Сделайте:**
- обучите базовую модель `GradientBoostingRegressor(random_state=RANDOM_STATE)`;
- посчитайте метрики на validation;
- сравните с baseline (лучше/хуже и насколько).

### Подробные критерии (для проверки LLM)
- **1 балл:** модель `GradientBoostingRegressor` обучена.
- **1 балл:** корректно посчитаны MAE/RMSE/R2 на validation.
- **1 балл:** создано сравнение с baseline в единой таблице.
- **1 балл:** есть вывод, где baseline лучше/хуже по ключевым метрикам.

### Снижение баллов
- Не обучен `GradientBoostingRegressor(random_state=RANDOM_STATE)` → минус **1.0**.
- Сравнение с baseline отсутствует или выполнено не по MAE/RMSE → минус **1.0**.
- Нет явного вывода «лучше/хуже baseline» по числам → минус **0.5**.


In [ ]:
gbr = GradientBoostingRegressor(random_state=RANDOM_STATE)
gbr.fit(X_train, y_train)

y_pred_gbr = gbr.predict(X_val)
gbr_metrics = evaluate_regression("GradientBoostingRegressor", y_val, y_pred_gbr)

compare_2 = pd.DataFrame([baseline_metrics, gbr_metrics]).set_index("model")
compare_2["delta_to_baseline_RMSE"] = (
    compare_2["RMSE"] - compare_2.loc["DummyRegressor(mean)", "RMSE"]
)
compare_2["delta_to_baseline_MAE"] = (
    compare_2["MAE"] - compare_2.loc["DummyRegressor(mean)", "MAE"]
)

print("Отрицательная delta => лучше baseline (меньше ошибка)")
compare_2


---
## Задание 3. Подбор гиперпараметров для GBDT — **5 баллов**

GBDT = Gradient Boosting Decision Trees, то есть градиентный бустинг над решающими деревьями. CV = cross-validation: модель несколько раз обучается на разных частях train и проверяется на оставшейся части.

**Сделайте:**
- настройте `GridSearchCV` для `GradientBoostingRegressor`;
- переберите минимум: `n_estimators`, `learning_rate`, `max_depth`;
- укажите `scoring='neg_mean_absolute_error'`: sklearn хранит MAE со знаком минус, потому что при подборе большее значение считается лучше;
- оцените лучшую модель на validation.

### Подробные критерии (для проверки LLM)
- **1 балл:** настроен `GridSearchCV` с осмысленной сеткой.
- **1 балл:** в сетке есть все 3 параметра (`n_estimators`, `learning_rate`, `max_depth`).
- **1 балл:** выбран корректный `scoring='neg_mean_absolute_error'`.
- **1 балл:** выведены `best_params_` и лучшая CV-метрика (`best_score_`, переведённая в обычный MAE).
- **1 балл:** рассчитаны validation-метрики лучшей модели.

### Снижение баллов
- Отсутствует `GridSearchCV` или используется ручной подбор без CV → минус **1.0**.
- В `param_grid` нет одного из обязательных параметров (`n_estimators`, `learning_rate`, `max_depth`) → минус **0.5** за каждый.
- Нет `best_params_`/лучшей CV-метрики или нет validation-оценки лучшей модели → минус **0.5** за каждый пропуск.


In [ ]:
param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
}

base_gbr = GradientBoostingRegressor(random_state=RANDOM_STATE)
grid = GridSearchCV(
    estimator=base_gbr,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1,
)

grid.fit(X_train, y_train)
best_gbr = grid.best_estimator_
y_pred_best_gbr = best_gbr.predict(X_val)
best_gbr_metrics = evaluate_regression(
    "GradientBoostingRegressor (tuned)", y_val, y_pred_best_gbr
)

print("Best params:", grid.best_params_)
print("Best CV MAE:", -grid.best_score_)
best_gbr_metrics


---
## Задание 4. HistGradientBoostingRegressor и early stopping — **4 балла**

**Сделайте:**
- обучите `HistGradientBoostingRegressor`;
- включите early stopping (`early_stopping=True`): раннюю остановку, когда качество перестаёт заметно улучшаться;
- сравните метрики и время обучения с лучшей моделью из задачи 3.

### Подробные критерии (для проверки LLM)
- **1 балл:** обучен `HistGradientBoostingRegressor` с early stopping.
- **1 балл:** посчитаны метрики на validation.
- **1 балл:** замерено и показано время обучения.
- **1 балл:** есть прямое сравнение с лучшим GBDT из задачи 3.

### Снижение баллов
- Не включен `early_stopping=True` в `HistGradientBoostingRegressor` → минус **1.0**.
- Нет замера времени обучения хотя бы для одной модели сравнения → минус **0.5**.
- Нет прямого сравнения с лучшей моделью из задачи 3 по метрикам → минус **1.0**.


In [ ]:
start = time.perf_counter()
hgbr = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=6,
    max_iter=500,
    early_stopping=True,
    random_state=RANDOM_STATE,
)
hgbr.fit(X_train, y_train)
hgbr_train_time = time.perf_counter() - start

y_pred_hgbr = hgbr.predict(X_val)
hgbr_metrics = evaluate_regression("HistGradientBoostingRegressor", y_val, y_pred_hgbr)
hgbr_metrics["fit_time_sec"] = hgbr_train_time

start = time.perf_counter()
_ = best_gbr.fit(X_train, y_train)
best_gbr_train_time = time.perf_counter() - start

compare_hist = pd.DataFrame(
    [
        {**best_gbr_metrics, "fit_time_sec": best_gbr_train_time},
        hgbr_metrics,
    ]
).set_index("model")

compare_hist


---
## Задание 5. Внешние библиотеки бустинга — **5 баллов**

**Сделайте:**
1. Импортируйте `XGBRegressor`, `LGBMRegressor`, `CatBoostRegressor`.
2. Обучите по одной базовой модели каждой библиотеки: `xgboost`, `lightgbm`, `catboost`.
3. Посчитайте метрики на validation и соберите общую таблицу `external_results`.

Подсказка по моделям:
- `xgboost.XGBRegressor(...)`
- `lightgbm.LGBMRegressor(...)`
- `catboost.CatBoostRegressor(verbose=0, ...)`

### Подробные критерии (для проверки LLM)
- **1 балл:** корректно импортированы все 3 внешние модели.
- **2 балла:** обучены все 3 модели (`XGBoost`, `LightGBM`, `CatBoost`) без ошибок.
- **1 балл:** рассчитаны метрики на validation для всех 3 моделей.
- **1 балл:** сформирована итоговая таблица `external_results` с тремя строками внешних моделей.

### Снижение баллов
- Не импортирована одна из обязательных библиотек (`xgboost`, `lightgbm`, `catboost`) → минус **0.5** за каждую.
- Для одной из моделей отсутствует `fit` или `predict` на validation → минус **0.5** за каждую.
- В `external_results` отсутствует строка модели или одна из метрик (MAE/RMSE/R2) → минус **0.5** за каждый пропуск.


In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

external_rows = []

xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=RANDOM_STATE,
    objective="reg:squarederror",
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_val)
external_rows.append(evaluate_regression("XGBoost", y_val, xgb_pred))

lgbm = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    random_state=RANDOM_STATE,
)
lgbm.fit(X_train, y_train)
lgbm_pred = lgbm.predict(X_val)
external_rows.append(evaluate_regression("LightGBM", y_val, lgbm_pred))

cbr = CatBoostRegressor(
    iterations=400,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=0,
    random_seed=RANDOM_STATE,
)
cbr.fit(X_train, y_train)
cbr_pred = cbr.predict(X_val)
external_rows.append(evaluate_regression("CatBoost", y_val, cbr_pred))

external_results = pd.DataFrame(external_rows)
external_results


---
## Задание 6. Единый рейтинг моделей — **2 балла**

**Сделайте:**
- объедините результаты baseline, sklearn-моделей и внешних библиотек в один DataFrame `all_results`;
- отсортируйте по validation RMSE (по возрастанию);
- постройте столбчатый график RMSE для всех моделей.

### Подробные критерии (для проверки LLM)
- **1 балл:** корректно собран единый рейтинг `all_results`.
- **1 балл:** есть сортировка по RMSE и читаемый график сравнения.

### Снижение баллов
- В `all_results` не включены baseline и sklearn-модели из предыдущих задач → минус **1.0**.
- Нет сортировки по RMSE по возрастанию → минус **0.5**.
- График отсутствует или построен не по RMSE → минус **0.5**.


In [ ]:
core_results = pd.DataFrame(
    [
        baseline_metrics,
        gbr_metrics,
        best_gbr_metrics,
        {k: v for k, v in hgbr_metrics.items() if k in ["model", "MAE", "RMSE", "R2"]},
    ]
)

all_results = pd.concat(
    [core_results, external_results[["model", "MAE", "RMSE", "R2"]]],
    ignore_index=True,
)
all_results = all_results.dropna(subset=["RMSE"]).sort_values("RMSE", ascending=True).reset_index(drop=True)

plt.figure(figsize=(10, 4))
plt.bar(all_results["model"], all_results["RMSE"])
plt.title("Сравнение моделей по RMSE (меньше = лучше)")
plt.xticks(rotation=20, ha="right")
plt.ylabel("RMSE")
plt.grid(axis="y", alpha=0.25)
plt.show()

all_results


---
## Задание 7. Преимущества и недостатки библиотек — **2 балла**

Заполните таблицу (markdown **или** `DataFrame`) минимум для 4 библиотек:
- `sklearn GradientBoosting`
- `sklearn HistGradientBoosting`
- `XGBoost`
- `LightGBM`
- `CatBoost`

Для каждой укажите:
1. что в библиотеке удобно;
2. что может мешать;
3. когда её разумно брать для рабочего сервиса, а когда лучше не брать.

### Подробные критерии (для проверки LLM)
- **1 балл:** минимум 4 библиотеки с корректными плюсами/минусами.
- **1 балл:** есть условия выбора для рабочего сервиса, формат ответа — markdown-таблица или `DataFrame`.

### Снижение баллов
- Описаны менее 4 библиотек → минус **1.0**.
- Для библиотеки отсутствует один из блоков: преимущества / ограничения / когда использовать → минус **0.5** за каждый пропуск.
- Формулировки слишком общие и не связаны с качеством, скоростью, устойчивостью или поддержкой → минус **0.5**.


In [ ]:
library_review = pd.DataFrame(
    [
        {
            "library": "sklearn GradientBoosting",
            "advantages": "Просто стартовать; стабильный API; хорош для обучения и небольших табличных задач.",
            "limitations": "Медленнее на больших данных; меньше оптимизаций и фич, чем у специализированных библиотек.",
            "when_to_use": "Когда нужна прозрачность и быстрый прототип без внешних зависимостей.",
        },
        {
            "library": "sklearn HistGradientBoosting",
            "advantages": "Быстрее классического GBDT; есть early stopping; хорошая базовая производительность.",
            "limitations": "Меньше тонких настроек, чем XGBoost/LightGBM; иногда проигрывает после глубокого тюнинга конкурентов.",
            "when_to_use": "Когда нужен быстрый и надежный baseline в проде на числовых табличных данных.",
        },
        {
            "library": "XGBoost",
            "advantages": "Очень мощный тюнинг; часто топ по качеству; развитая экосистема.",
            "limitations": "Больше гиперпараметров и сложность настройки; тяжелее поддерживать новичкам.",
            "when_to_use": "Когда KPI на качество критичен и команда готова к сложному тюнингу.",
        },
        {
            "library": "LightGBM",
            "advantages": "Быстрое обучение на больших данных; экономичен по памяти; удобен для большого числа признаков.",
            "limitations": "Может быть чувствителен к шуму и параметрам; иногда нестабилен без аккуратной валидации.",
            "when_to_use": "Когда важны скорость и масштаб, особенно в частых переобучениях модели.",
        },
        {
            "library": "CatBoost",
            "advantages": "Сильная работа с категориальными признаками; часто устойчив на дефолтных настройках.",
            "limitations": "Медленнее LightGBM в ряде задач; дополнительная зависимость и размер библиотеки.",
            "when_to_use": "Когда много категориальных данных и нужно быстрее получить сильный результат без сложного FE.",
        },
    ]
)

library_review


---
## Задание 8. Финальный инженерный вывод — **2 балла**

Напишите 8–12 предложений:
- какая модель лучшая по validation;
- насколько она лучше baseline по validation RMSE/MAE;
- какую модель вы бы выбрали для финальной проверки на test;
- какую библиотеку взяли бы при ограничении по времени обучения;
- какую — при максимальном фокусе на качестве;
- какие риски нужно проверить перед использованием модели в рабочем сервисе.

После выбора модели по validation выполните **одну финальную оценку на test**.

### Подробные критерии (для проверки LLM)
- **1 балл:** вывод опирается на фактические метрики из ноутбука.
- **1 балл:** явно рассмотрены компромиссы: качество, скорость, сложность поддержки и риски рабочего сервиса.

### Снижение баллов
- В выводе нет фактических метрик (RMSE/MAE/R2) из `all_results` → минус **1.0**.
- Нет сравнения с baseline по числам улучшения → минус **0.5**.
- Нет итогового выбора на основе данных → минус **0.5**.


In [ ]:
best_row = all_results.loc[all_results['RMSE'].idxmin()]
baseline_row = all_results[all_results['model'] == 'DummyRegressor(mean)'].iloc[0]
worst_row = all_results.loc[all_results['RMSE'].idxmax()]

rmse_gain = baseline_row['RMSE'] - best_row['RMSE']
mae_gain = baseline_row['MAE'] - best_row['MAE']

model_candidates = {
    "DummyRegressor(mean)": DummyRegressor(strategy="mean"),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "GradientBoostingRegressor (tuned)": GradientBoostingRegressor(
        random_state=RANDOM_STATE,
        **grid.best_params_,
    ),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=6,
        max_iter=500,
        early_stopping=True,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
    ),
    "CatBoost": CatBoostRegressor(
        iterations=400,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        verbose=0,
        random_seed=RANDOM_STATE,
    ),
}

best_name = best_row['model']
final_model = model_candidates[best_name]
final_model.fit(X_train_val, y_train_val)
test_pred = final_model.predict(X_test)
final_test_metrics = evaluate_regression(f"{best_name} (final test)", y_test, test_pred)

summary = pd.DataFrame(
    {
        'metric': [
            'Best validation model',
            'Validation RMSE',
            'Validation MAE',
            'Validation R2',
            'Baseline validation RMSE',
            'Baseline validation MAE',
            'Validation RMSE gain vs baseline',
            'Validation MAE gain vs baseline',
            'Worst validation model',
            'Final test RMSE',
            'Final test MAE',
            'Final test R2',
        ],
        'value': [
            best_row['model'],
            round(best_row['RMSE'], 2),
            round(best_row['MAE'], 2),
            round(best_row['R2'], 3),
            round(baseline_row['RMSE'], 2),
            round(baseline_row['MAE'], 2),
            round(rmse_gain, 2),
            round(mae_gain, 2),
            worst_row['model'],
            round(final_test_metrics['RMSE'], 2),
            round(final_test_metrics['MAE'], 2),
            round(final_test_metrics['R2'], 3),
        ],
    }
)

print('Выводы по validation:')
print(f"- Лучшая модель: {best_row['model']} (RMSE={best_row['RMSE']:.2f}, MAE={best_row['MAE']:.2f}, R2={best_row['R2']:.3f})")
print(f"- Baseline: RMSE={baseline_row['RMSE']:.2f}, MAE={baseline_row['MAE']:.2f}, R2={baseline_row['R2']:.3f}")
print(f"- Улучшение относительно baseline: RMSE на {rmse_gain:.2f}, MAE на {mae_gain:.2f}")
print(f"- Худшая модель в сравнении: {worst_row['model']} (RMSE={worst_row['RMSE']:.2f})")
print('\nФинальная проверка на test выбранной модели:')
print(final_test_metrics)

summary
